# 4 Mark5.3 라벨 없음 0 5회 · 결과물 4개

**언제 돌리나:** `2 Mark5.3 본실험 40회` 의 [셀 8] 이 끝난 뒤(고정 규칙 5회까지)입니다. 그 노트북의 [셀 9] 는 정답 200·라벨 없음 0 의 5회가 빠져 멈춥니다(아래 이유).
**L4 로 실행하십시오.** '모두 실행' 한 번으로 끝까지 돕니다. 세션이 끊기면 이 노트북을 다시 '모두 실행' 하면 됩니다(끝난 회차는 건너뜁니다).

**왜 따로 돌리나 (2026-10-26)**
- 정답 200·라벨 없음 0 의 5회는 학습을 시작하자마자 `유효한 unlabeled 샘플이 0개입니다` 로 멈췄습니다. 10-22 판이 데이터 검사를 '라벨 없는 쪽도 0개면 멈춤' 으로 바꿨기 때문입니다.
- `model/train_mark5.py` 는 정답 라벨 쪽이 0개일 때만 멈추게 되돌렸습니다(10-22 이전 동작). 바뀐 줄은 라벨 없는 클립이 0개일 때만 실행됩니다.
- `train_mark5.py` 는 회차 신원 해시에 들어가는 파일입니다. 그래서 이 노트북은 그 5회만 돌립니다(`--cmb_unlabeled_n 0`). 이미 끝난 35회는 다시 돌리지 않고, 결과 파일이 다 있는지만 확인합니다.
- **`2 Mark5.3 본실험 40회` 를 다시 '모두 실행' 하지 마십시오.** 고친 코드에서는 35회가 '남은 회차' 로 잡혀 다시 학습합니다.

| 단계 | 셀 |
|---|---|
| 런타임 · 드라이브 · 코드 판 · 패키지 | [셀 1~4] |
| 계획 확인: 라벨 없음 0 의 5회만 · 나머지 35회 결과 파일 | [셀 5] |
| 학습 → val → test 5회 | [셀 6] |
| 5회 완료 확인 | [셀 7] |
| 결과물 4개 (비교모델 60회 평가 — `5 비교모델 평가만 60회(argmax)` — 가 끝나길 기다림) | [셀 8] |

결과물: `plots/four_results_test.md` · `plots/four_results_{1,2,3,4}_test.csv` · `plots/four_results_{1,2}_test.png`

In [ ]:
#========================================================================
# [셀 1] 런타임 확인 — L4 권장 · Blackwell(sm_100·120)이면 멈춤 · CPU 는 허용(멈추지 않음)
#========================================================================
import subprocess, sys
_probe = subprocess.run([sys.executable, "-c",
    "import os, torch\n"
    "ok = torch.cuda.is_available()\n"
    "cap = torch.cuda.get_device_capability(0) if ok else None\n"
    "print('GPU', torch.cuda.get_device_name(0) if ok else '없음 — CPU 로 돈다')\n"
    "print('SM', cap[0] * 10 + cap[1] if cap else 'cpu')\n"
    "print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| CPU 코어', os.cpu_count())"],
    capture_output=True, text=True)
print(_probe.stdout, _probe.stderr[-2000:])
_sm = next((l.split()[1] for l in _probe.stdout.splitlines() if l.startswith("SM ")), "cpu")
if _sm in ("100", "120"):
    raise SystemExit("[STOP] Blackwell(sm_100/sm_120) 런타임입니다. L4 로 바꿔 다시 '모두 실행' 하십시오(2026-10-17 G4 계산 오류).")
!nvidia-smi --query-gpu=name,driver_version,compute_cap,memory.total --format=csv 2>/dev/null || echo "nvidia-smi 없음(CPU 런타임)"
!free -g | head -2

In [ ]:
#========================================================================
# [셀 2] 구글 드라이브 마운트
#========================================================================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#========================================================================
# [셀 3] 작업 폴더 · 데이터 목록 · 드라이브 코드가 2026-10-26(라벨 없음 0 수정) 판인가
#========================================================================
import csv, hashlib, json, os, re, time
EXP = "/content/drive/MyDrive/NewProject/codes/experiments"
ROOT = f"{EXP}/Mark5.3"
if not os.path.isdir(ROOT):
    raise SystemExit(f"[STOP] 코드 폴더가 없습니다: {ROOT}")
os.chdir(ROOT)
sys.path.insert(0, f"{ROOT}/model")


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


for name, n_expected in {"dataset_labeled.csv": 2000, "dataset_val.csv": 430,
                         "dataset_unlabeled.csv": 2000, "dataset_test.csv": 430}.items():
    if not os.path.isfile(name):
        raise SystemExit(f"[STOP] 데이터 목록이 없습니다: {name}")
    with open(name, newline="", encoding="utf-8-sig") as f:
        n = sum(1 for _ in csv.DictReader(f))
    print(name, n)
    if n != n_expected:
        raise SystemExit(f"[STOP] {name} 행 수 {n}, 기대 {n_expected}")
NEED = {"model/import_teachers.py": ['choices=["segment", "mn10"]'],
        "model/make_teacher_scores.py": ['"--teacher_kind"', "mark53_audio_sha256"],
        "model/teacher_gate_report.py": ["C4_raise_off_mean"],
        "model/make_combiner_labels.py": ['"--scores"'],
        "model/label_freshness.py": ["def check_scores", "def check_pseudo_csv", "def audio_manifest"],
        # [2026-10-26] 논문 결과물 4개용 8조합(--cmb_paper) · 결과물 스크립트
        "model/run_experiments.py": ["CMB_PLAN = [", "CMB_PAPER_PLAN = [", '"--cmb_paper"', '"--cmb_unlabeled_n"'],
        # [2026-10-26] 결과물: 비교모델도 보정 없는 argmax(예측 CSV 의 PreBias_Prob_*) · 빠진 회차면 멈춤
        "model/make_four_results.py": ["def main", '"--baseline_root"', "def baseline_argmax", "PreBias_Prob_", '"--allow_missing"'],
        "model/verify_release.py": ['"--refresh_source"'],
        # [2026-10-25 / A안] 옵티마이저 Adam · 학습률 선정값 고정(새 층·백본 1e-4) · run_tag _ADAM (10-22~10-25 Prodigy 를 걷어낸 판)
        "vild/vild_config.py": ['self.optimizer_name = "adam"', 'tag += "_ADAM"'],
        # [2026-10-26] 라벨 없음 0 회차가 멈추지 않는 판(정답 라벨 쪽이 0개일 때만 멈춤)
        "model/train_mark5.py": ["optimizer = optim.Adam(optimizer_params", "if is_labeled and len(self.samples) == 0:"]}
miss = [(f, s) for f, ss in NEED.items() for s in ss
        if not os.path.isfile(f) or s not in open(f, encoding="utf-8").read()]
if miss:
    raise SystemExit(f"[STOP] 드라이브 코드가 10-26(라벨 없음 0 수정) 판이 아닙니다: {miss}")
size = os.path.getsize("weights/mn10_as_mAP_471.pt") if os.path.isfile("weights/mn10_as_mAP_471.pt") else None
print("mn10 가중치", size, "bytes (기대 19708753)")
if size != 19708753:
    raise SystemExit("[STOP] weights/mn10_as_mAP_471.pt 가 없거나 크기가 다릅니다")
print("코드 판 확인 완료")

In [ ]:
#========================================================================
# [셀 4] 패키지 — 코랩의 GPU torch·torchvision 은 그대로 둔다 · 문장 인코더 네 패키지만 판 고정
#========================================================================
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                "sentence-transformers==2.7.0", "transformers==4.46.3",
                "huggingface-hub==0.36.2", "tokenizers==0.20.3"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "librosa==0.10.2.post1", "soundfile==0.12.1",
                "pydub==0.25.1", "thop==0.1.1.post2209072238"], check=True)
import sentence_transformers, transformers, huggingface_hub, tokenizers
got = {"sentence-transformers": sentence_transformers.__version__, "transformers": transformers.__version__,
       "huggingface-hub": huggingface_hub.__version__, "tokenizers": tokenizers.__version__}
want = {"sentence-transformers": "2.7.0", "transformers": "4.46.3", "huggingface-hub": "0.36.2", "tokenizers": "0.20.3"}
print(got)
if got != want:
    raise SystemExit(f"[STOP] 패키지 판 불일치 {got} — 런타임 → 세션 다시 시작 → 다시 '모두 실행'")
import shutil
from pathlib import Path
HF_SRC = Path("/content/drive/MyDrive/NewProject/hf_cache/models--sentence-transformers--all-MiniLM-L6-v2")
HF_HUB = Path.home() / ".cache" / "huggingface" / "hub"
if not HF_SRC.is_dir():
    raise SystemExit(f"[STOP] 문장 인코더 캐시가 없습니다: {HF_SRC}")
HF_HUB.mkdir(parents=True, exist_ok=True)
shutil.copytree(HF_SRC, HF_HUB / HF_SRC.name, dirs_exist_ok=True)
os.environ.update({"HF_HOME": str(HF_HUB.parent), "HF_HUB_CACHE": str(HF_HUB),
                   "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"})
from sentence_transformers import SentenceTransformer
SentenceTransformer("all-MiniLM-L6-v2")
print("문장 인코더 오프라인 캐시 준비 완료")

In [ ]:
#========================================================================
# [셀 5] 계획 확인 — 라벨 없음 0 의 5회만 돌리는가 · 나머지 35회 결과 파일이 다 있는가 (아무것도 안 돌림)
#========================================================================
RUN_U0 = "--mark_version mark5.2 --preset cmb --cmb_paper --cmb_unlabeled_n 0 --seeds 1 2 3 4 5"
RUN_ALL = "--mark_version mark5.2 --preset cmb --cmb_paper --seeds 1 2 3 4 5"


def plan(run):
    p = subprocess.run(f"{sys.executable} model/run_experiments.py {run} --dry_run", shell=True, capture_output=True, text=True)
    m = re.search(r"완료 (\d+)개 / 남은 (\d+)개", p.stdout)
    if p.returncode != 0 or not m:
        print(p.stdout[-3000:]); print(p.stderr[-3000:])
        raise SystemExit("[STOP] 실행기 계획(--dry_run)을 읽지 못했습니다 — 위 출력 확인")
    if "[STALE]" in p.stdout:
        print(p.stdout[-3000:])
        raise SystemExit("[STOP] 옛 판 가짜 정답·교사 캐시가 있습니다")
    todo = [l.strip() for l in p.stdout.splitlines() if "[실행]" in l]
    tags = re.findall(r"\[(?:실행|건너뜀)\][^\n]*?\s(mark5\.2_\S+)", p.stdout)
    return int(m.group(1)), int(m.group(2)), todo, tags


done, left, todo, u0_tags = plan(RUN_U0)
print(f"라벨 없음 0 계획 {len(u0_tags)}회 · 완료 {done} · 남은 {left}")
# 실행기 dry_run 은 [건너뜀] 줄을 먼저 찍으므로 seed 순서는 보지 않는다(앞 seed 하나만 실패한 뒤 다시 돌려도 통과하게)
_seeds = sorted(int(t.rsplit("_s", 1)[1]) for t in u0_tags if re.search(r"_ADAM_l200_u0_s\d+$", t))
if len(u0_tags) != 5 or len(set(u0_tags)) != 5 or _seeds != [1, 2, 3, 4, 5]:
    raise SystemExit(f"[STOP] 계획이 정답 200·라벨 없음 0 의 seed 1~5 가 아닙니다: {u0_tags}")
_, _, _, all_tags = plan(RUN_ALL)
others = [t for t in all_tags if t not in u0_tags]
if len(all_tags) != 40 or len(set(all_tags)) != 40 or len(others) != 35:
    raise SystemExit(f"[STOP] 논문 결과물 40회 태그를 읽지 못했습니다({len(all_tags)}회)")
# 나머지 35회: 신원 해시(코드 판)는 보지 않고, 실행기와 같은 기준으로 '요약 완전 + 지금 체크포인트로 평가한 결과' 인지만 본다
import run_experiments as runner
from output_policy import matches_plain_policy
bad = []
for t in others:
    a = runner._artifacts("mark5.2", t)
    if not all(runner._summary_is_complete(a[s]) and matches_plain_policy(a[f"{s}_policy"], a["ckpt"], t, s) for s in ("val", "test")):
        bad.append(t)
if bad:
    print("\n".join(bad[:20]))
    raise SystemExit(f"[STOP] 나머지 35회 중 결과가 완전하지 않은 것이 {len(bad)}개입니다 — '2 Mark5.3 본실험 40회' [셀 8] 이 끝났는지 확인")
print("나머지 35회 val·test 결과가 모두 지금 체크포인트로 평가한 완전한 결과입니다 — 이 노트북은 라벨 없음 0 의 5회만 돌립니다")

In [ ]:
#========================================================================
# [셀 6] 정답 200·라벨 없음 0 ★ 5회 — 회차마다 학습 → val → test. 끝난 회차는 건너뛴다. 시간 제한 없음
#========================================================================
!python model/run_experiments.py {RUN_U0}

In [ ]:
#========================================================================
# [셀 7] 확인 — 5회가 전부 끝났는가 (남았으면 멈춘다)
#========================================================================
done, left, todo, _ = plan(RUN_U0)
if left:
    print("\n".join(todo))
    raise SystemExit(f"[STOP] {left}회가 끝나지 않았습니다 — logFiles/<태그>__*.log 확인 뒤 다시 '모두 실행'(끝난 회차는 건너뜁니다)")
print("정답 200·라벨 없음 0 의 5회 전부 완료")

In [ ]:
#========================================================================
# [셀 8] 결과물 4개 — 비교모델 60회 평가('5 비교모델 평가만 60회(argmax)' 노트북, 판정 규칙 argmax_v1)가 끝나길 기다린 뒤 만든다 (시간 제한 없음)
#========================================================================
DATA_ROOT = f"{EXP}/Mark5.2"
BASELINES = ("Baseline_EfficientAT_mn10", "Baseline_PANNs_CNN6", "Baseline_CED_Tiny")
B_RUN = "--phase main --conds sup --labeled_ns 0 1000 500 200 --seeds 1 2 3 4 5"   # 정답 라벨만 × 정답 4수준 × seed 1~5 = 20회


def baseline_left(repo):
    p = subprocess.run(f"{sys.executable} run_experiments.py {B_RUN} --data_root {DATA_ROOT} --dry_run",
                       shell=True, capture_output=True, text=True, cwd=f"{EXP}/{repo}")
    rows = [l for l in p.stdout.splitlines() if "[실행]" in l or "[건너뜀]" in l]
    if p.returncode != 0 or len(rows) != 20:
        return None
    return sum("[실행]" in l for l in rows)


t0 = time.time()
while True:
    left = {repo: baseline_left(repo) for repo in BASELINES}
    # [2026-10-26] 비교모델 eval 은 성능 요약을 먼저 쓰고 예측 CSV 를 나중에 쓴다 — 결과물이 읽는 test 예측 CSV 60개까지 있어야 끝난 것으로 본다
    pred_missing = [f"{repo}/{tag}" for repo, mtag in zip(BASELINES, ("efficientat_mn10", "panns_cnn6", "ced_tiny"))
                    for l in ("", "_l1000", "_l500", "_l200") for s in (1, 2, 3, 4, 5)
                    for tag in [f"{mtag}_sup_adam{l}_s{s}"]
                    if not os.path.isfile(f"{EXP}/{repo}/plots/prediction_details_{tag}.csv")]
    if all(v == 0 for v in left.values()) and not pred_missing:
        break
    print(time.strftime("%H:%M:%S"), f"(경과 {(time.time() - t0) / 60:.0f}분) 비교모델 남은 회차:", left, flush=True)
    time.sleep(300)
subprocess.run([sys.executable, "-u", "model/make_four_results.py", "--baseline_root", EXP], check=True)
print("완료 — plots/four_results_test.md · four_results_{1,2,3,4}_test.csv · four_results_{1,2}_test.png")